# Data Preparation for Statistical Analysis

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd().resolve().parents[1]  # repository root (run from 02-papers/Result-01)
DATASETS_DIR = (Path("/home/ec2-user/studies") / "hpp_datasets").resolve()

sys.path.append(str(PROJECT_DIR))

popu_pfile = DATASETS_DIR / "population" / "population.parquet"
anthro_pfile = DATASETS_DIR / "anthropometrics" / "anthropometrics.parquet"

OUTPUT_PREFIX = 'outputs'
FIGURE_PREFIX = 'figures'

## Omics

In [ ]:
pfile ='../01-interim_data/omics_data/lipidomics_with_metadata.parquet'
df_lipid = pd.read_parquet(pfile)
print(df_lipid.shape)

metadata_col_lipid = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_lipid = [col for col in metadata_col_lipid if col in df_lipid.columns]
df_lipid_feature = df_lipid.drop(columns=metadata_col_lipid)
df_lipid_meta = df_lipid[metadata_col_lipid]

# ══════════════════════════════════════════════════════════════
pfile ='../01-interim_data/omics_data/gut_microbiome_with_metadata.parquet'
df_gut = pd.read_parquet(pfile)
print(df_gut.shape)

metadata_col_gut = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_gut = [col for col in metadata_col_gut if col in df_gut.columns]
df_gut_feature = df_gut.drop(columns=metadata_col_gut)
df_gut_meta = df_gut[metadata_col_gut]

# ══════════════════════════════════════════════════════════════
pfile ='../01-interim_data/omics_data/transcriptomics_with_metadata.parquet'
df_rna = pd.read_parquet(pfile)
print(df_rna.shape)

metadata_col_rna = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_rna = [col for col in metadata_col_rna if col in df_rna.columns]
df_rna_feature = df_rna.drop(columns=metadata_col_rna)
df_rna_meta = df_rna[metadata_col_rna]

# ══════════════════════════════════════════════════════════════
pfile ='../01-interim_data/omics_data/metabolomics_with_metadata.parquet'
df_metabo = pd.read_parquet(pfile)
print(df_metabo.shape)

metadata_col_metabo = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_metabo = [col for col in metadata_col_metabo if col in df_metabo.columns]
df_metabo_feature = df_metabo.drop(columns=metadata_col_metabo)
df_metabo_meta = df_metabo[metadata_col_metabo]

# ══════════════════════════════════════════════════════════════
pfile ='../01-interim_data/omics_data/oral_microbiome_with_metadata.parquet'
df_oral = pd.read_parquet(pfile)
print(df_oral.shape)

metadata_col_oral = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'age', 'year_of_birth', 'month_of_birth', 'sex', 'collection_datetime', 'batch', 'bmi']
metadata_col_oral = [col for col in metadata_col_oral if col in df_oral.columns]
df_oral_feature = df_oral.drop(columns=metadata_col_oral)
df_oral_meta = df_oral[metadata_col_oral]

# Sex distribution

In [ ]:
def make_meta(df, omics_name,
              id_col="participant_id", rs_col="research_stage", age_col="age", sex_col="sex", bmi_col="bmi"):
    return (df[[id_col, rs_col, age_col, sex_col, bmi_col]]
            .rename(columns={id_col:"id", rs_col:"research_stage", age_col:"age", sex_col:"sex", bmi_col:"bmi"})
            .assign(omics=omics_name))

meta_all = pd.concat([
    make_meta(df_lipid,  "lipid"),
    make_meta(df_gut,    "gut"),
    make_meta(df_rna,    "rna"),
    make_meta(df_metabo, "metabo"),
    make_meta(df_oral,   "oral"),
], ignore_index=True)

meta_all = meta_all.drop_duplicates(subset=["id", "research_stage"]).reset_index(drop=True)

In [ ]:
# Age groups
bins = [40, 45, 50, 55, 60, 65, 70]
labels = ['40-45', '45-50', '50-55', '55-60', '60-65', '65-70']

meta_all['age_group'] = pd.cut(meta_all['age'], bins=bins, labels=labels, right=False)

# Count women and men per age group
result = meta_all.groupby(['age_group', 'sex'], observed=True).size().unstack(fill_value=0)
result.columns = ['female (sex=0)', 'male (sex=1)']
result.index.name = 'age_group'

result.to_csv(f'{OUTPUT_PREFIX}/age_sex_distribution.csv')
print(result)

## Donut

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib import font_manager
import glob, os

for f in glob.glob(os.path.expanduser("~/.fonts/*.ttf")):
    font_manager.fontManager.addfont(f)
font_manager.findfont("Arial", fallback_to_default=False)

plt.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.5,
})

# ─── DATA ───────────────────────────────────────
age_sex_distri = pd.read_csv(f'{OUTPUT_PREFIX}/age_sex_distribution.csv')
female = age_sex_distri['female (sex=0)'].tolist()
male   = age_sex_distri['male (sex=1)'].tolist()
male_pct   = sum(male) / (sum(female) + sum(male)) * 100
female_pct = 100 - male_pct

# ─── STYLE ──────────────────────────────────────
color_male   = "#4878CF"
color_female = "#D65F5F"
gap_deg      = 4.0
ring_w       = 0.40
start_deg    = 90
LIM          = 1.08            # half-width of the axes

CM    = 1 / 2.54
FIG_W = 4.5 * CM               # inches
FIG_H = 4.5 * CM

# ─── Font sizes derived from figure size (Nature: 5–7 pt; panel labels 8 pt) ──
NAT_MIN, NAT_MAX = 5.0, 7.0

def clamp(v, lo, hi):
    return float(np.clip(round(v * 2) / 2, lo, hi))   # 0.5 pt steps

# Points per data unit (equal aspect; the smaller of width and height)
pt_per_unit = min(FIG_W, FIG_H) * 72 / (2 * LIM)
ring_pt     = ring_w * pt_per_unit                # ring thickness (pt)
hole_pt     = 2 * (1 - ring_w) * pt_per_unit      # hole diameter (pt)

FS_PCT   = clamp(ring_pt * 0.28, NAT_MIN, NAT_MAX)   # percentages on the ring: about 1/3 of ring thickness
FS_NAME  = clamp(hole_pt / 14,   NAT_MIN, NAT_MAX)   # "Female" (6 characters) must fit in half the hole
FS_ICON  = clamp(hole_pt / 9,    NAT_MIN, 10)        # symbols may be slightly larger, as graphical elements
FS_PANEL = 8                                          # Nature panel labels: 8 pt bold
LW       = max(0.25, min(1.0, ring_pt * 0.02))        # white sector separators ≥ 0.25 pt

print(f"ring={ring_pt:.1f}pt hole={hole_pt:.1f}pt -> "
      f"pct={FS_PCT} name={FS_NAME} icon={FS_ICON} lw={LW:.2f}")

# ─── FIGURE ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor("white")
ax.set_aspect("equal")
ax.axis("off")

available  = 360 - gap_deg * 2
ang_male   = male_pct   / 100 * available
ang_female = female_pct / 100 * available

sectors = [
    (start_deg,                      ang_male,   color_male,   f"{male_pct:.0f}%"),
    (start_deg + ang_male + gap_deg, ang_female, color_female, f"{female_pct:.0f}%"),
]

for theta_start, ang, color, pct_label in sectors:
    ax.add_patch(mpatches.Wedge(
        center=(0, 0), r=1.0,
        theta1=theta_start, theta2=theta_start + ang,
        width=ring_w, facecolor=color, edgecolor="white", linewidth=LW,
    ))
    mid = np.radians(theta_start + ang / 2)
    r_label = 1.0 - ring_w / 2
    ax.text(r_label * np.cos(mid), r_label * np.sin(mid), pct_label,
            ha="center", va="center",
            fontsize=FS_PCT, fontweight="bold", color="white")

# ─── CENTRE TEXT (vertical position scales with font size to avoid overlap) ──
x_off   = (1 - ring_w) * 0.42
icon_y  =  (FS_ICON * 0.45) / pt_per_unit
name_y  = -(FS_NAME * 0.95) / pt_per_unit

for x, sym, name, col in [(-x_off, "♂", "Male",   color_male),
                          ( x_off, "♀", "Female", color_female)]:
    ax.text(x, icon_y, sym,  ha="center", va="center", fontsize=FS_ICON, color=col)
    ax.text(x, name_y, name, ha="center", va="center", fontsize=FS_NAME, color="#555555")

ax.set_xlim(-LIM, LIM)
ax.set_ylim(-LIM, LIM)
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

os.makedirs("figures", exist_ok=True)
plt.savefig("figures/donut_sex.pdf", facecolor="white")
plt.savefig("figures/donut_sex.png", dpi=600, facecolor="white")
print("Done")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.path import Path
import matplotlib.patches as patches
import numpy as np

# ══════════════════════════════════════════════════════════════════════════════
# DATA
# ══════════════════════════════════════════════════════════════════════════════
age_groups    = ["40-45","45-50","50-55","55-60","60-65","65-70"]
male_counts   = [1228, 1404, 1195,  834,  566, 405]
female_counts = [1131, 1466, 1343, 1075,  749, 514]
x = np.arange(len(age_groups))

FONT   = "Arial"
C_MALE = "#5B7EC9"
C_FEM  = "#E0648A"
CM     = 1 / 2.54
FIG_W  = 5. * CM
FIG_H  = 5.2 * CM
BAR_W  = 0.55

# ── Font and element sizes (pt, at final print size) ──
FS_TICK  = 5      # tick labels
FS_LABEL = 5.5    # axis titles
FS_TITLE = 6      # plot title
FS_TAG   = 5.5    # Male / Female labels
DOT_S    = 7      # marker area
LW       = 0.4    # axis and tick line width

from matplotlib import font_manager
import glob, os

# Register Arial from ~/.fonts
for f in glob.glob(os.path.expanduser("~/.fonts/*.ttf")):
    font_manager.fontManager.addfont(f)

# Fail if Arial (regular or bold) is missing instead of falling back silently
font_manager.findfont(font_manager.FontProperties(family=FONT), fallback_to_default=False)
font_manager.findfont(font_manager.FontProperties(family=FONT, weight="bold"),
                      fallback_to_default=False)

plt.rcParams.update({
    "font.family":       FONT,
    "font.size":         FS_TICK,
    "axes.linewidth":    LW,
    "xtick.major.width": LW,
    "ytick.major.width": LW,
    "xtick.major.size":  1.5,
    "ytick.major.size":  1.5,
    "xtick.major.pad":   1.5,
    "ytick.major.pad":   1.5,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

fig, (ax_m, ax_f) = plt.subplots(2, 1, figsize=(FIG_W, FIG_H), sharex=True)
fig.patch.set_facecolor("white")

# Fixed margins: the exported size is exactly 5.0 × 4.1 cm
# Must be set before drawing rounded bars, which depend on the final axes position
L, R, B, T = 0.17, 0.98, 0.17, 0.91
fig.subplots_adjust(left=L, right=R, bottom=B, top=T, hspace=0.12)


def rounded_rect_path(x, y, w, h, r):
    """Path of a rectangle with four rounded corners (Bézier curves)."""
    k = 0.5523
    rx = ry = r
    verts = [
        (x + rx, y), (x + w - rx, y),
        (x + w - rx + k*rx, y), (x + w, y + ry - k*ry), (x + w, y + ry),
        (x + w, y + h - ry),
        (x + w, y + h - ry + k*ry), (x + w - rx + k*rx, y + h), (x + w - rx, y + h),
        (x + rx, y + h),
        (x + rx - k*rx, y + h), (x, y + h - ry + k*ry), (x, y + h - ry),
        (x, y + ry),
        (x, y + ry - k*ry), (x + rx - k*rx, y), (x + rx, y),
        (x + rx, y),
    ]
    codes = [Path.MOVETO, Path.LINETO,
             Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.LINETO,
             Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.LINETO,
             Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.LINETO,
             Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
    return Path(verts, codes)


def draw_rounded_bars(ax, xpos, values, color, bar_w):
    fig = ax.get_figure()
    ax_pos = ax.get_position()
    fig_w_px, fig_h_px = fig.get_size_inches() * fig.dpi
    ax_w_px = ax_pos.width  * fig_w_px
    ax_h_px = ax_pos.height * fig_h_px
    px_per_xunit = ax_w_px / np.ptp(ax.get_xlim())
    px_per_yunit = ax_h_px / np.ptp(ax.get_ylim())

    r_px = bar_w * 0.3 * px_per_xunit
    rx = r_px / px_per_xunit
    ry = r_px / px_per_yunit

    for xi, val in zip(xpos, values):
        r_use = min(rx, bar_w * 0.45, ry, val * 0.45)
        ax.add_patch(patches.PathPatch(
            rounded_rect_path(xi - bar_w / 2, 0, bar_w, val, r_use),
            facecolor=color, edgecolor="none", linewidth=0,
            zorder=3, clip_on=True,
        ))


def style_ax(ax):
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, linestyle=":", linewidth=0.3, color="#cccccc", zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", labelsize=FS_TICK)
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3, integer=True))


def panel(ax, counts, color, tag):
    style_ax(ax)
    ymax = max(counts) * 1.30
    ax.set_ylim(0, ymax)
    ax.set_xlim(-0.6, len(age_groups) - 0.4)
    draw_rounded_bars(ax, x, counts, color, BAR_W)
    ax.scatter(x, [v + ymax * 0.05 for v in counts],
               s=DOT_S, color=color, edgecolors="white",
               linewidths=0.3, zorder=6, clip_on=False)
    ax.text(0.98, 0.97, tag, transform=ax.transAxes,
            ha="right", va="top", fontsize=FS_TAG,
            color=color, fontweight="bold")


# ── MALE / FEMALE ────────────────────────────────────────────────────────────
panel(ax_m, male_counts,   C_MALE, "Male")
panel(ax_f, female_counts, C_FEM,  "Female")

ax_m.set_title("Age distribution", fontsize=FS_TITLE, pad=3)
ax_f.set_xticks(x)
ax_f.set_xticklabels(age_groups, fontsize=FS_TICK)
ax_f.set_xlabel("Age group (years)", fontsize=FS_LABEL, labelpad=2)

# One shared y-axis label for both panels
fig.text(0.015, (B + T) / 2, "Number of individuals",
         rotation=90, ha="left", va="center", fontsize=FS_LABEL)

# No bbox_inches="tight": it would crop the canvas and change the 5.0 × 4.1 cm size
plt.savefig("figures/pyramid_sex.pdf", dpi=300, facecolor="white")
plt.savefig("figures/pyramid_sex.png", dpi=600, facecolor="white")
print("Saved.")

# Sample Time

In [ ]:
df_rna_meta_bar = df_rna[['participant_id', 'research_stage', 'collection_date']].copy()
df_rna_meta_bar['Source'] = 'Transcriptomics'
df_metabolo_meta_bar = df_metabo[['participant_id', 'research_stage', 'collection_date']].copy()
df_metabolo_meta_bar['Source'] = 'Metabolomics'
df_lipid_meta_bar = df_lipid[['participant_id', 'research_stage', 'collection_date']].copy()
df_lipid_meta_bar['Source'] = 'Lipidomics'
df_gut_meta_bar = df_gut[['participant_id', 'research_stage', 'collection_date']].copy()
df_gut_meta_bar['Source'] = 'Gut Microbiome'
df_oral_meta_bar = df_oral[['participant_id', 'research_stage', 'collection_date']].copy()
df_oral_meta_bar['Source'] = 'Oral Microbiome'
# Convert 'collection_date' of df_oral_meta_bar to nanosecond precision
df_oral_meta_bar['collection_date'] = df_oral_meta_bar['collection_date'].astype('datetime64[ns]')

In [ ]:
# Data frames to concatenate
all_dfs_bar = [
    df_rna_meta_bar, 
    df_metabolo_meta_bar, 
    df_lipid_meta_bar, 
    df_gut_meta_bar, 
    df_oral_meta_bar
]

df_combined_bar = pd.concat(all_dfs_bar)

print(df_combined_bar.head())
print(f"Total rows after concatenation: {len(df_combined_bar)}")

In [ ]:
import pandas as pd

# df_combined_bar['collection_date'] = pd.to_datetime(df_combined_bar['collection_date'])

# 1. Count samples per two-week period (freq='2W-SUN': bins ending on Sunday) and Source
time_binned_counts = df_combined_bar.groupby([
    pd.Grouper(key='collection_date', freq='2W-SUN'), 
    'Source'
]).size()

# 2. Pivot 'Source' to columns (wide format for plotting); missing counts become 0
summary_df = time_binned_counts.unstack(level='Source').fillna(0)

# 3. Convert the date index to integers for plotting (optional)
summary_df = summary_df.astype(int)

print("Preview of the processed data:")
print(summary_df.head())

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import font_manager


# ================================================================
# Data preparation
# ================================================================

summary_df = summary_df.copy()
summary_df.index = pd.to_datetime(summary_df.index)
summary_df = summary_df.sort_index()

os.makedirs(FIGURE_PREFIX, exist_ok=True)


# ================================================================
# Size and fonts (final print size; Nature: text 5–7 pt, panel labels 8 pt bold)
# ================================================================

MM    = 1 / 25.4
FIG_W = 110 * MM
FIG_H = 52 * MM

NAT_MIN   = 5.0    # Nature minimum font size
FS_TICK   = 6
FS_LABEL  = 7
FS_LEGEND = 6
FS_PANEL  = 8
PANEL_LABEL = None  # None = no panel label

LW     = 0.5       # axis and tick line width (Nature: ≥ 0.25 pt)
PAD_PT = 2.0       # minimum distance between text and canvas edge (pt)
GAP_PT = 4.0       # gap between legend and plot area (pt)


# ================================================================
# Fonts and global style
# ================================================================

for f in glob.glob(os.path.expanduser("~/.fonts/*.ttf")):
    font_manager.fontManager.addfont(f)
font_manager.findfont("Arial", fallback_to_default=False)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": FS_TICK,
    "axes.labelsize": FS_LABEL,
    "xtick.labelsize": FS_TICK,
    "ytick.labelsize": FS_TICK,
    "axes.linewidth": LW,
    "xtick.major.width": LW,
    "ytick.major.width": LW,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.major.pad": 2.5,
    "ytick.major.pad": 2.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ================================================================
# Colours
# ================================================================

NC_COLORS = {
    "Transcriptomics": "#CB6E70",
    "Metabolomics":    "#68AADD",
    "Lipidomics":      "#54AD94",
    "Gut Microbiome":  "#DDA062",
    "Oral Microbiome": "#896EAE",
}

missing_colors = [c for c in summary_df.columns if c not in NC_COLORS]
if missing_colors:
    raise ValueError(f"No colour defined in NC_COLORS for columns: {missing_colors}")
bar_colors = [NC_COLORS[c] for c in summary_df.columns]


# ================================================================
# Canvas
# ================================================================

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

summary_df.plot(
    kind="bar", stacked=True, color=bar_colors, width=0.75,
    linewidth=0, edgecolor="none", ax=ax, legend=False, zorder=3,
)


# ================================================================
# X axis: one major label every six months
# ================================================================

major_ticks, major_dates = [], []
seen = set()
for i, date in enumerate(summary_df.index):
    key = (date.year, 1 if date.month <= 6 else 2)
    if key not in seen:
        seen.add(key)
        major_ticks.append(i)
        major_dates.append(date)

def make_labels(step):
    """Label every step-th major tick; add the year at its first occurrence."""
    labels, last_year = [], None
    for k, d in enumerate(major_dates):
        if k % step:
            labels.append("")
            continue
        if d.year != last_year:
            labels.append(f"{d.strftime('%b')}\n{d.year}")
            last_year = d.year
        else:
            labels.append(d.strftime("%b"))
    return labels

ax.set_xticks(major_ticks)
ax.set_xticklabels(make_labels(1), rotation=0, ha="center", va="top", linespacing=1.1)
ax.tick_params(axis="x", which="major", length=3, width=LW, pad=2.5, color="black")

ax.set_xticks(range(len(summary_df)), minor=True)
ax.tick_params(axis="x", which="minor", length=1.5, width=0.3,
               color="#999999", bottom=True, labelbottom=False)

ax.set_xlabel("Collection date", labelpad=3)
ax.set_xlim(-0.6, len(summary_df) - 0.4)


# ================================================================
# Y axis
# ================================================================

ax.set_ylabel("Number of samples", labelpad=3)
ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5, integer=True))
ax.yaxis.grid(True, linestyle="-", linewidth=0.3, color="#cccccc", zorder=0)
ax.xaxis.grid(False)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# ================================================================
# Helper: pt → pixels
# ================================================================

renderer = fig.canvas.get_renderer()
px = lambda pt: pt * fig.dpi / 72
fig_w_px, fig_h_px = fig.bbox.width, fig.bbox.height


# ================================================================
# Panel label
# ================================================================

panel_w_px = 0
if PANEL_LABEL:
    panel_txt = fig.text(px(PAD_PT) / fig_w_px, 1 - px(PAD_PT) / fig_h_px,
                         PANEL_LABEL, ha="left", va="top",
                         fontsize=FS_PANEL, fontweight="bold")
    panel_w_px = panel_txt.get_window_extent(renderer).width + px(PAD_PT)


# ================================================================
# Legend: title and entries on one line, centred on the figure, width fitted automatically
# ================================================================

ORDER = ["Transcriptomics", "Metabolomics", "Lipidomics",
         "Gut Microbiome", "Oral Microbiome"]
handles = [plt.Rectangle((0, 0), 1, 1, facecolor=NC_COLORS[c], edgecolor="none")
           for c in ORDER]

def make_legend(fs, colsp, ncol):
    leg = fig.legend(
        handles=handles, labels=ORDER, title="Molecule class",
        loc="upper center",
        bbox_to_anchor=(0.5, 1 - px(PAD_PT) / fig_h_px),
        ncol=ncol, frameon=False,
        prop={"size": fs}, title_fontsize=fs,
        handlelength=0.7, handleheight=0.7,
        handletextpad=0.4, columnspacing=colsp,
        borderaxespad=0, borderpad=0,
    )
    # Move the title to the far left of the entry row
    title_box, entries_box = leg._legend_box.get_children()
    leg._legend_box._children = [entries_box]
    entries_box._children = [title_box] + entries_box.get_children()
    entries_box.align = "center"
    return leg

# Available width: leave symmetric room for the panel label so the legend stays centred
avail_px = fig_w_px - 2 * (px(PAD_PT) + panel_w_px)

candidates = [                     # (font size, column spacing, columns), in order of preference
    (FS_LEGEND, 1.0, len(ORDER)),
    (FS_LEGEND, 0.6, len(ORDER)),
    (5.5,       0.6, len(ORDER)),
    (NAT_MIN,   0.6, len(ORDER)),
    (FS_LEGEND, 1.0, 3),           # still too wide → two rows
]
for fs, colsp, ncol in candidates:
    leg = make_legend(fs, colsp, ncol)
    if leg.get_window_extent(renderer).width <= avail_px:
        break
    leg.remove()
print(f"Legend: fontsize={fs} pt, columnspacing={colsp}, ncol={ncol}")


# ================================================================
# Margins from actual text extents: nothing is clipped and the size is exactly FIG_W × FIG_H
# ================================================================

def fit_axes():
    leg_bottom = leg.get_window_extent(renderer).y0
    top = (leg_bottom - px(GAP_PT)) / fig_h_px
    left, right, bottom = 0.10, 0.98, 0.15
    for _ in range(4):
        ax.set_position([left, bottom, right - left, top - bottom])
        tb = ax.get_tightbbox(renderer)     # includes tick labels and axis titles
        left   += (px(PAD_PT) - tb.x0) / fig_w_px
        bottom += (px(PAD_PT) - tb.y0) / fig_h_px
        right  -= (tb.x1 - (fig_w_px - px(PAD_PT))) / fig_w_px

fit_axes()


# ================================================================
# De-overlap x labels: show every other label when the gap is < 1.5 pt
# ================================================================

def labels_overlap():
    boxes = [t.get_window_extent(renderer)
             for t in ax.get_xticklabels() if t.get_text()]
    return any(b2.x0 - b1.x1 < px(1.5) for b1, b2 in zip(boxes, boxes[1:]))

step = 1
while labels_overlap() and step < len(major_ticks):
    step += 1
    ax.set_xticklabels(make_labels(step), rotation=0, ha="center",
                       va="top", linespacing=1.1)
if step > 1:
    print(f"X labels: every {step} half-years shown")
    fit_axes()

pos = ax.get_position()
cx  = pos.x0 + pos.width / 2                       # centre of the plot area (figure coordinates)

leg_w = leg.get_window_extent(renderer).width / fig_w_px
pad_x = px(PAD_PT) / fig_w_px
cx = min(max(cx, pad_x + leg_w / 2), 1 - pad_x - leg_w / 2)

leg.set_bbox_to_anchor((cx, 1 - px(PAD_PT) / fig_h_px),
                       transform=fig.transFigure)

# ================================================================
# Export (no bbox_inches="tight", so the size is exactly 110 × 52 mm)
# ================================================================

png_path = f"{FIGURE_PREFIX}/sample_collection_bar.png"
pdf_path = f"{FIGURE_PREFIX}/sample_collection_bar.pdf"

fig.savefig(png_path, dpi=600, facecolor="white")
fig.savefig(pdf_path, facecolor="white")

plt.show()
plt.close(fig)

print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")